# 05 - Curated Methanol Reaction Path on Cu(211)

This notebook asks a focused chemistry question:

**Where does the dataset contain a credible surface reaction path toward
methanol, and which calculations should be discarded before we trust that
story?**

The point is not to force the entire database into a single mechanism. The
point is to perform a careful, thesis-like selection:

1. start from the full HDF archive,
2. remove calculations that are clearly unphysical or incomplete,
3. isolate methanol-relevant intermediates,
4. identify one surface family with an interpretable sequence of states,
5. visualize the thermodynamic and constrained-optimization evidence.

In [ ]:
from pathlib import Path
from collections import Counter
import math
import re
import subprocess
import sys
import tempfile

sys.path.insert(0, r"/Users/dk2994/Desktop/git/DFTDataFrame/src")
import DFTDataFrame.Tools as OP

pd = OP.pd
np = OP.np
plt = OP.plt

DATA_PATH = Path(r"/Users/dk2994/Desktop/Uni/scripts/created_frame.hdf")
DFTDATAFRAME_SRC = Path(r"/Users/dk2994/Desktop/git/DFTDataFrame/src")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12


def load_onepiece_hdf(path=DATA_PATH, key="df"):
    """Load the local OnePiece-style HDF table.

    This notebook uses the local DFTDataFrame package as the available
    OnePiece-compatible analysis layer. The actual HDF payload is read through
    the pandas namespace exposed by that package.
    """
    path = Path(path)
    try:
        return OP.pd.read_hdf(path, key=key).copy()
    except Exception as original_error:
        helper_python = Path("/Users/dk2994/opt/anaconda3/bin/python")
        if not helper_python.exists():
            raise original_error
        output = Path(tempfile.NamedTemporaryFile(delete=False, suffix=".pkl", prefix="created_frame_").name)
        script = """
from pathlib import Path
import sys
import numpy as np
import pandas as pd
try:
    import numpy.core as numpy_core
    sys.modules.setdefault("numpy._core", numpy_core)
    sys.modules.setdefault("numpy._core.multiarray", np.core.multiarray)
    sys.modules.setdefault("numpy._core.numeric", np.core.numeric)
    import ase.constraints  # noqa: F401
except Exception:
    pass
source = Path(sys.argv[1])
key = sys.argv[2]
target = Path(sys.argv[3])
pd.read_hdf(source, key=key).to_pickle(target)
"""
        completed = subprocess.run(
            [str(helper_python), "-c", script, str(path), key, str(output)],
            capture_output=True,
            text=True,
            check=False,
        )
        if completed.returncode != 0:
            detail = completed.stderr.strip() or completed.stdout.strip()
            raise RuntimeError(f"Could not read {path}. Helper reader failed: {detail}") from original_error
        return OP.pd.read_pickle(output).copy()


ADSORBATE_TOKENS = [
    "CH3OH", "CH3O", "HCOOH", "H2COOH", "HCOO", "COOH", "CO2", "HCO", "CO"
]


def guess_adsorbate(name):
    if not isinstance(name, str):
        return ""
    for token in sorted(ADSORBATE_TOKENS, key=len, reverse=True):
        if re.search(rf"(^|[-_%]){re.escape(token)}($|[-_%])", name):
            return token
    return ""


def guess_record_class(name, path=""):
    text = f"{name} {path}".lower()
    if "gasphases" in text:
        return "gas"
    if "copt" in text:
        return "copt"
    if "convergence" in text:
        return "convergence"
    if "bulk" in text:
        return "bulk"
    if "clean" in text:
        return "clean_surface"
    if guess_adsorbate(name):
        return "adsorbate"
    return "other"


def guess_facet(name):
    if not isinstance(name, str):
        return ""
    for facet in ["100", "110", "111", "211", "221"]:
        if facet in name:
            return facet
    return ""


def material_family(name):
    if not isinstance(name, str):
        return "unknown"
    token = name.split("-")[0]
    return token or "unknown"


def surface_key_from_name(name):
    if not isinstance(name, str):
        return ""
    key = name
    key = re.sub(r"-copt-.*$", "", key)
    key = re.sub(r"-(CH3OH|CH3O|HCOOH|H2COOH|HCOO|COOH|CO2|HCO|CO)([-_%].*)?$", "", key)
    key = re.sub(r"-[0-9]+$", "", key)
    return key


def add_taxonomy(df):
    out = df.copy()
    out["record_class"] = [guess_record_class(n, p) for n, p in zip(out["Name"], out["Path"], strict=False)]
    out["facet"] = out["Name"].map(guess_facet)
    out["material_family"] = out["Name"].map(material_family)
    out["adsorbate"] = out["Name"].map(guess_adsorbate)
    out["surface_key"] = out["Name"].map(surface_key_from_name)
    return out


def formula_counts(formula):
    counts = {}
    if not isinstance(formula, str):
        return counts
    for element, number in re.findall(r"([A-Z][a-z]?)(\d*)", formula):
        counts[element] = counts.get(element, 0) + int(number or 1)
    return counts


def add_formula_features(df):
    out = df.copy()
    parsed = out["Formula"].map(formula_counts)
    all_elements = sorted({el for counts in parsed for el in counts})
    for el in all_elements:
        out[el] = parsed.map(lambda counts: counts.get(el, 0))
    out["n_elements"] = parsed.map(len)
    out["n_atoms_formula"] = parsed.map(lambda counts: sum(counts.values()))
    return out


def gas_reference_energy(df, token):
    pattern = rf"^gasphases-{re.escape(token)}(?:$|[-_].*)"
    subset = df[df["Name"].astype(str).str.contains(pattern, regex=True, case=False, na=False)]
    subset = subset[pd.to_numeric(subset["E"], errors="coerce").notna()]
    if subset.empty:
        return np.nan
    return float(subset["E"].iloc[0])


def assign_clean_references(df):
    out = df.copy()
    clean = out[out["record_class"] == "clean_surface"].copy()
    clean = clean[pd.to_numeric(clean["E"], errors="coerce").notna()]
    clean_map = clean.groupby("surface_key")["E"].min().to_dict()
    clean_name_map = clean.sort_values("E").drop_duplicates("surface_key").set_index("surface_key")["Name"].to_dict()
    out["clean_ref_E"] = out["surface_key"].map(clean_map)
    out["clean_ref_name"] = out["surface_key"].map(clean_name_map)
    out["delta_E_surface"] = pd.to_numeric(out["E"], errors="coerce") - pd.to_numeric(out["clean_ref_E"], errors="coerce")
    return out


def adsorption_energy_conventions(df):
    out = df.copy()
    e_co = gas_reference_energy(out, "CO")
    e_h2 = gas_reference_energy(out, "H2")
    e_ch3oh = gas_reference_energy(out, "CH3OH")
    e_hcooh = gas_reference_energy(out, "HCOOH")
    out["E_ads_CO"] = np.where(out["adsorbate"].eq("CO"), out["E"] - out["clean_ref_E"] - e_co, np.nan)
    out["E_ads_CH3O_from_CH3OH"] = np.where(
        out["adsorbate"].eq("CH3O"),
        out["E"] - out["clean_ref_E"] - e_ch3oh + 0.5 * e_h2,
        np.nan,
    )
    out["E_ads_HCOO_from_HCOOH"] = np.where(
        out["adsorbate"].eq("HCOO"),
        out["E"] - out["clean_ref_E"] - e_hcooh + 0.5 * e_h2,
        np.nan,
    )
    out["E_ads_COOH_from_HCOOH"] = np.where(
        out["adsorbate"].eq("COOH"),
        out["E"] - out["clean_ref_E"] - e_hcooh + 0.5 * e_h2,
        np.nan,
    )
    return out


df_raw = load_onepiece_hdf()
df = add_formula_features(add_taxonomy(df_raw))
df["E"] = pd.to_numeric(df["E"], errors="coerce")
df["fmax"] = pd.to_numeric(df["fmax"], errors="coerce")
df["has_structure"] = df["struc"].map(lambda value: value.__class__.__name__ == "Atoms")
df["has_contcar"] = df["CONTCAR"].map(lambda value: value.__class__.__name__ == "Atoms")

## 1. Build a transparent curation table

In [ ]:
METHANOL_TOKENS = [
    "CO2", "HCOO", "HCOO_H", "HCOOH", "H2COOH",
    "H2CO_OH", "H2CO_H", "CH3O", "CH3O_H", "H3COH", "OCH3OH"
]
SURFACE_STATE_TOKENS = [
    "OCH3OH", "CH3O_H", "H3COH", "CH3O", "H2CO_OH", "H2CO_H",
    "H2COOH_H", "H2COOH", "HCOOH_H", "HCOOH", "HCOO_H", "HCOO",
    "COOH", "CO2", "HCO_OH", "HCO_H", "HCO", "H2COO"
]


def is_bad_formula(value):
    text = "" if value is None else str(value).strip()
    return text in {"", "0", "nan", "None"}


def curation_flags(frame):
    out = frame.copy()
    out["flag_missing_energy"] = out["E"].isna()
    out["flag_zero_energy"] = out["E"].eq(0)
    out["flag_bad_formula"] = out["Formula"].map(is_bad_formula)
    out["flag_name_test"] = out["Name"].astype(str).str.contains(r"test", case=False, na=False)
    out["flag_name_convergence"] = out["Name"].astype(str).str.contains(r"convergence", case=False, na=False)
    out["flag_missing_structure"] = ~out["has_structure"] & ~out["has_contcar"]
    out["flag_high_fmax_static"] = out["record_class"].ne("copt") & out["fmax"].gt(0.05)
    out["flag_high_fmax_copt"] = out["record_class"].eq("copt") & out["fmax"].gt(0.10)
    out["flag_any_bad"] = out[
        [
            "flag_missing_energy",
            "flag_zero_energy",
            "flag_bad_formula",
            "flag_name_test",
            "flag_name_convergence",
            "flag_missing_structure",
            "flag_high_fmax_static",
            "flag_high_fmax_copt",
        ]
    ].any(axis=1)
    return out


def infer_surface_reference_name(name):
    text = "" if name is None else str(name)
    if "gasphases" in text or "convergence" in text or "test" in text:
        return ""
    if "-copt-" in text:
        return text.split("-copt-")[0]
    for token in SURFACE_STATE_TOKENS:
        marker = f"-{token}"
        if marker in text:
            return text.split(marker)[0]
    return text


curated = curation_flags(assign_clean_references(df))
curated["surface_ref_guess"] = curated["Name"].map(infer_surface_reference_name)
surface_reference_rows = curated[
    curated["Name"].eq(curated["surface_ref_guess"]) & ~curated["flag_any_bad"]
][["Name", "E"]].drop_duplicates("Name")
surface_reference_map = surface_reference_rows.set_index("Name")["E"].to_dict()
curated["surface_ref_E"] = curated["surface_ref_guess"].map(surface_reference_map)
curated["relative_to_surface_ref_eV"] = curated["E"] - curated["surface_ref_E"]

summary = pd.Series(
    {
        "all_rows": len(curated),
        "removed_rows": int(curated["flag_any_bad"].sum()),
        "kept_rows": int((~curated["flag_any_bad"]).sum()),
        "static_high_fmax_removed": int(curated["flag_high_fmax_static"].sum()),
        "copt_high_fmax_removed": int(curated["flag_high_fmax_copt"].sum()),
        "test_name_removed": int(curated["flag_name_test"].sum()),
        "bad_formula_removed": int(curated["flag_bad_formula"].sum()),
    }
)
summary

These rules are intentionally conservative.

- Static minima should be well relaxed, so we ask for `fmax <= 0.05`.
- Constrained images can be a bit rougher, so we allow `fmax <= 0.10`.
- Rows with `E = 0`, missing energy, missing formula, `test`, or
  `convergence` in the name are removed.

This is exactly the kind of curation logic that belongs in a thesis chapter:
explicit, reproducible, and open to discussion.

In [ ]:
flag_counts = curated.filter(like="flag_").sum().sort_values(ascending=False)
flag_counts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
flag_counts.drop(labels=["flag_any_bad"]).sort_values().plot(kind="barh", ax=ax, color="#e45756")
ax.set_title("How many rows are removed by each curation rule?")
ax.set_xlabel("rows")
ax.set_ylabel("curation flag")
plt.tight_layout()
plt.show()

## 2. Keep only methanol-relevant intermediates

In [ ]:
methanol_mask = curated["Name"].astype(str).str.contains("|".join(METHANOL_TOKENS), regex=True, na=False)
methanol = curated[methanol_mask & ~curated["flag_any_bad"]].copy()

print("curated methanol-related rows:", len(methanol))
methanol["adsorbate_family"] = methanol["Name"].astype(str).map(
    lambda name: next((token for token in METHANOL_TOKENS if token in name), "")
)
methanol[["Name", "record_class", "material_family", "facet", "adsorbate_family", "E", "fmax"]].head(20)

In [ ]:
family_counts = (
    methanol.groupby(["material_family", "facet"])
    .size()
    .sort_values(ascending=False)
    .head(12)
    .rename("rows")
)
family_counts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
labels = [f"{material}-{facet or 'na'}" for material, facet in family_counts.index]
ax.barh(labels[::-1], family_counts.to_numpy()[::-1], color="#72b7b2")
ax.set_title("Where does the methanol-related chemistry live?")
ax.set_xlabel("curated rows")
ax.set_ylabel("material / facet")
plt.tight_layout()
plt.show()

At this point a clear case study emerges. The Cu(211) branch is especially
useful because it contains:

- static intermediates such as `HCOO_H`, `HCOOH`, `H2COOH`, `H2CO_OH`,
  `H2CO_H`, `CH3O`, `CH3O_H`, and `H3COH`,
- plus several explicit `copt` trajectories that connect neighboring states.

That is enough to tell a chemically meaningful story toward methanol.

## 3. Focus on the Cu(211) family

In [ ]:
cu211 = methanol[
    methanol["Name"].astype(str).str.contains(r"^Cu-211-", regex=True, na=False)
].copy()

cu211["surface_branch"] = cu211["Name"].astype(str).map(
    lambda name: "Cu-211-Ga" if name.startswith("Cu-211-Ga")
    else ("Cu-211-Zn" if name.startswith("Cu-211-Zn") else "Cu-211-clean")
)

branch_counts = cu211["surface_branch"].value_counts()
branch_counts

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
branch_counts.sort_values().plot(kind="barh", ax=ax, color="#4c78a8")
ax.set_title("Curated methanol-related rows inside the Cu(211) subset")
ax.set_xlabel("rows")
ax.set_ylabel("surface branch")
plt.tight_layout()
plt.show()

## 4. Build thermodynamic anchors from static intermediates

In [ ]:
STATIC_SEQUENCE = [
    "CO2", "HCOO_H", "HCOOH", "H2COOH", "H2CO_OH", "H2CO_H", "CH3O", "CH3O_H", "H3COH"
]


def first_matching_token(name, tokens):
    for token in tokens:
        if token in str(name):
            return token
    return ""


static_cu211 = cu211[
    cu211["record_class"].ne("copt") & cu211["adsorbate_family"].isin(STATIC_SEQUENCE)
].copy()
static_cu211["state"] = static_cu211["Name"].map(lambda name: first_matching_token(name, STATIC_SEQUENCE))
static_cu211["relative_to_surface_ref_eV"] = static_cu211["relative_to_surface_ref_eV"]

anchors = (
    static_cu211
    .groupby(["surface_branch", "state"], as_index=False)
    .agg(
        n_structures=("Name", "size"),
        best_name=("Name", "first"),
        best_energy=("relative_to_surface_ref_eV", "min"),
        best_fmax=("fmax", "min"),
    )
)

anchors["state"] = pd.Categorical(anchors["state"], categories=STATIC_SEQUENCE, ordered=True)
anchors = anchors.sort_values(["surface_branch", "state"])
anchors.head(30)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for branch, branch_df in anchors.groupby("surface_branch"):
    ordered = branch_df.dropna(subset=["best_energy"]).sort_values("state")
    x = [STATIC_SEQUENCE.index(state) for state in ordered["state"].astype(str)]
    ax.plot(x, ordered["best_energy"], marker="o", linewidth=2, label=branch)

ax.set_xticks(range(len(STATIC_SEQUENCE)))
ax.set_xticklabels(STATIC_SEQUENCE, rotation=45, ha="right")
ax.set_ylabel(r"$E - E_{clean}$ / eV")
ax.set_title("Best curated Cu(211) intermediates along a methanol-oriented state sequence")
ax.legend()
plt.tight_layout()
plt.show()

This plot is not a full free-energy diagram. It is a **curated energy map of
available static intermediates**. That distinction matters.

A thesis should say clearly: these points tell us which intermediates are in
the database and how stable they are relative to the matching bare surface
reference for the same Cu(211) branch.
They do **not** by themselves guarantee a unique catalytic mechanism.

## 5. Parse the constrained-optimization trajectories that point toward methanol

In [ ]:
def parse_copt_name(name):
    if not isinstance(name, str) or "copt" not in name:
        return pd.Series({"pathway": "", "image": np.nan, "state_pair": "", "state_left": "", "state_right": ""})
    image_match = re.search(r"-(\d{2})$", name)
    image = int(image_match.group(1)) if image_match else np.nan
    state_match = re.search(r"copt-([^%]+)%(.+?)(?:-\d{2})$", name)
    if state_match:
        left = state_match.group(1)
        right = state_match.group(2)
        pair = left + " -> " + right
    else:
        left = ""
        right = ""
        pair = ""
    pathway = re.sub(r"-\d{2}$", "", name)
    return pd.Series({"pathway": pathway, "image": image, "state_pair": pair, "state_left": left, "state_right": right})


cu211_copt = cu211[cu211["record_class"].eq("copt")].copy()
cu211_copt = pd.concat([cu211_copt, cu211_copt["Name"].apply(parse_copt_name)], axis=1)

target_pairs = cu211_copt[
    cu211_copt["state_pair"].astype(str).str.contains(r"HCOO|HCOOH|H2COOH|H2CO_OH|H2CO_H|CH3O|H3COH", regex=True, na=False)
].copy()

pair_counts = target_pairs["state_pair"].value_counts()
pair_counts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
pair_counts.sort_values().plot(kind="barh", ax=ax, color="#f58518")
ax.set_title("Cu(211) constrained transformations that feed the methanol discussion")
ax.set_xlabel("rows")
ax.set_ylabel("state pair")
plt.tight_layout()
plt.show()

## 6. Plot the most complete Cu(211) image sequences

In [ ]:
profiles = (
    target_pairs.dropna(subset=["image", "E"])
    .groupby(["surface_branch", "pathway", "state_pair"], as_index=False)
    .agg(n_images=("image", "size"), min_image=("image", "min"), max_image=("image", "max"))
)
profiles = profiles[profiles["n_images"] >= 5].sort_values(["surface_branch", "n_images"], ascending=[True, False])
profiles.head(20)

In [ ]:
top_profiles = profiles.head(6)
fig, axes = plt.subplots(len(top_profiles), 1, figsize=(10, 3.2 * len(top_profiles)), sharex=False)
if len(top_profiles) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, top_profiles.iterrows(), strict=False):
    prof = target_pairs[target_pairs["pathway"].eq(row["pathway"])].sort_values("image")
    ax.plot(prof["image"], prof["E"], marker="o", color="#54a24b")
    ax.set_title(f"{row['surface_branch']}: {row['state_pair']}")
    ax.set_ylabel("E / eV")
    ax.set_xlabel("image index")

plt.tight_layout()
plt.show()

## 7. A compact methanol-pathway reading

The curated Cu(211) subset supports the following chemistry-oriented reading:

1. **formate and formic-acid chemistry is present**:
   `HCOO_H`, `HCOOH`, and `H2COOH` are all available, with explicit `copt`
   links between neighboring states.
2. **deeper hydrogenation chemistry is also present**:
   `H2CO_OH`, `H2CO_H`, `CH3O`, `CH3O_H`, and `H3COH` occur as static states
   and partially as pathway images.
3. **the most complete methanol-endgame images are in Cu(211)**:
   - `H2CO_H -> CH3O`
   - `CH3O_H -> H3COH`
4. **the dataset is not perfectly uniform**:
   some branches are rich in statics, others in `copt`, and some promising
   pathways still have sparse coverage.

That is actually a realistic thesis conclusion. Good scientific curation does
not hide gaps; it identifies the mechanism-relevant parts of the archive and
states clearly where the evidence is strong and where it remains incomplete.

## 8. Export a table of the curated methanol pathway evidence

In [ ]:
evidence_table = target_pairs[
    ["Name", "surface_branch", "surface_ref_guess", "state_pair", "image", "E", "relative_to_surface_ref_eV", "fmax"]
].sort_values(["surface_branch", "state_pair", "image"])
evidence_table.head(40)